# מטלה בלמידת מכונה - ניתוח טקסט (סיווג רגשות של ביקורות IMDB)

**שם הסטודנט/ית:** אנג'לה י. 9456

### פרומפטים ועזרים ב-AI

**כלי:** Claude.

**פרומפטים בהם נעזרתי:**<br> 
"תשפר את הקוד שיהיה קריא יותר"<br>
"תמיר את קוד הפייתון המצורף למחברת ג'ופיטר בצורה הכי פשוטה שאפשר."<br>

**מטרת השימוש:** קבלת עזרה בכתיבת מחברת מהקוד, ניסוח טוב יותר של הקוד שכתבתי ובניסוח ההסברים במחברת.

### הבעיה וה-Dataset

המטלה עוסקת בסיווג רגשות של ביקורות סרטים (חיובי ושלילי), מתוך מאגר:<br> [IMDB Dataset of 50K Movie Reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews)<br> 
המאגר כולל 50,000 ביקורות טקסט באנגלית, מאוזן בין 25,000 ביקורות חיוביות ו-25,000 שליליות.<br> המטרה היא לאמן מודל סיווג בינארי (classification)<br> שיחזה מתוך הביקורות, האם הרגש חיובי או שלילי.

In [3]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

### טעינת ה-Dataset

Kaggle-המאגר המקורי מגיע מ <br> train/test-מגיע כקובץ יחיד, ללא חלוקה מובנית ל <br> לכן מתבצעת כאן חלוקה **חד-פעמית וקבועה** (80% train, 20% test, מאוזנת לפי התווית) - חלוקה זו לא תבוצע שוב.

In [4]:
df = pd.read_csv("IMDB Dataset.csv")
df["label"] = (df["sentiment"] == "positive").astype(int)

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df["label"], random_state=42
)
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Trainset shape:", train_df.shape)
print("Testset shape:", test_df.shape)

Trainset shape: (40000, 3)
Testset shape: (10000, 3)


In [5]:
train_df.head(5)

,review,sentiment,label
0,I caught this little gem totally by accident b...,positive,1
1,I can't believe that I let myself into this mo...,negative,0
2,*spoiler alert!* it just gets to me the nerve ...,negative,0
3,If there's one thing I've learnt from watching...,negative,0
4,"I remember when this was in theaters, reviews ...",negative,0


In [6]:
test_df.head(5)

,review,sentiment,label
0,"Yes, MTV there really is a way to market Daria...",negative,0
1,The story of the bride fair is an amusing and ...,negative,0
2,"A team varied between Scully and Mulder, two o...",positive,1
3,This was a popular movie probably because of t...,negative,0
4,This movie made me so angry!! Here I am thinki...,negative,0


### מדד איכות

:זו בעיית סיווג בינארית עם שתי מחלקות מאוזנות, ללא מחלקה מרכזית אחת. לכן, מדד האיכות בו נשתמש הוא <br>
**macro-average F1**.

## חלק 2 - Feature Engineering

מבצעים ניקוי טקסט בסיסי (הסרת תגים וסימנים שאינם אותיות, המרה לאותיות קטנות), ולאחר מכן ממירים את הטקסט לוקטור מספרי בשיטת<br>
**TF-IDF** (Term Frequency - Inverse Document Frequency)<br>
את מדד התדירות מתאימים רק על נתוני האימון, ולאחר מכן משתמשים בו כדי להמיר גם את נתוני האימון וגם את נתוני הבדיקה.

In [7]:
def clean_text(text):
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    return text.lower()

train_df["clean_review"] = train_df["review"].apply(clean_text)
test_df["clean_review"] = test_df["review"].apply(clean_text)

In [8]:
vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")
X_train = vectorizer.fit_transform(train_df["clean_review"])
X_test = vectorizer.transform(test_df["clean_review"])
y_train = train_df["label"].values
y_test = test_df["label"].values

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (40000, 5000)
X_test shape: (10000, 5000)


### דוגמאות ל-Feature Engineering (train ו-test)

In [9]:
def show_feature_engineering(df, X, vectorizer, indices):
    feature_names = np.array(vectorizer.get_feature_names_out())
    for i in indices:
        row = X[i].toarray().ravel()
        top_idx = row.argsort()[::-1][:5]
        print("Original:", df["review"][i][:120], "...")
        print("Cleaned :", df["clean_review"][i][:120], "...")
        print("Top TF-IDF terms:", list(zip(feature_names[top_idx], row[top_idx].round(3))))
        print("-" * 80)

print(">>> Train examples")
show_feature_engineering(train_df, X_train, vectorizer, [0, 1, 2])

print(">>> Test examples")
show_feature_engineering(test_df, X_test, vectorizer, [0, 1, 2])

>>> Train examples
Original: I caught this little gem totally by accident back in 1980 or '81. I was at a revival theatre to see two old silly sci-fi ...
Cleaned : i caught this little gem totally by accident back in      or      i was at a revival theatre to see two old silly sci fi ...
Top TF-IDF terms: [('funnier', np.float64(0.28)), ('theatre', np.float64(0.255)), ('showed', np.float64(0.236)), ('sci', np.float64(0.226)), ('fi', np.float64(0.226))]
--------------------------------------------------------------------------------
Original: I can't believe that I let myself into this movie to accomplish a favor my friends ask me early this April 14, 2007. Thi ...
Cleaned : i can t believe that i let myself into this movie to accomplish a favor my friends ask me early this april           thi ...
Top TF-IDF terms: [('failed', np.float64(0.266)), ('theater', np.float64(0.245)), ('matter', np.float64(0.217)), ('april', np.float64(0.185)), ('accomplish', np.float64(0.182))]
--------------

## חלק 3 - מימוש אלגוריתם למידה: Logistic Regression
### Logistic Regression

הוא אלגוריתם לסיווג לינארי **Logistic Regression**
<br>
תחילה מחשבים שילוב לינארי של המאפיינים
<br>
לאחר מכן מעבירים את התוצאה דרך פונקציית **הסיגמואיד**
<br>

פונקציית הסיגמואיד מחזירה ערך בין 0 ל־1, שאותו ניתן לפרש כהסתברות שהדוגמה שייכת למחלקה החיובית.

**Gradient Descent** – האימון מתבצע באמצעות ירידת גרדיאנט. <br>
**Log Loss** – בכל איטרציה מחשבים את ההפרש בין התחזית של המודל לבין התווית האמיתית, ולאחר מכן מעדכנים את המשקלים כך שהשגיאה תקטן. <br>
**L2 Regularization** – משמשת לצמצום משקלים גדולים מדי ולסיוע במניעת התאמת יתר.


 שניתן לכוונן הם**Hyperparameters** ה-:

* `learning_rate` – קצב הלמידה.
* `n_iterations` – מספר האיטרציות באימון.
* `lambda_reg` – עוצמת רגולריזציית L2.


In [10]:
class LogisticRegressionScratch:
    def __init__(self, learning_rate=0.5, n_iterations=300, lambda_reg=0.01):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.lambda_reg = lambda_reg
        self.weights = None
        self.bias = 0.0

    @staticmethod
    def _sigmoid(z):
        z = np.clip(z, -30, 30)
        return 1.0 / (1.0 + np.exp(-z))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        y = np.asarray(y)

        for _ in range(self.n_iterations):
            linear = X.dot(self.weights) + self.bias
            y_pred = self._sigmoid(linear)
            error = y_pred - y

            grad_w = X.T.dot(error) / n_samples + (self.lambda_reg / n_samples) * self.weights
            grad_b = error.sum() / n_samples

            self.weights -= self.learning_rate * grad_w
            self.bias -= self.learning_rate * grad_b
        return self

    def predict_proba(self, X):
        linear = X.dot(self.weights) + self.bias
        return self._sigmoid(linear)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

## חלק 4 - אימון (הפעלת ה-flow)
בשלב זה מגדירים את המודל עם ערכי הפרמטרים שנבחרו, ולאחר מכן מאמנים אותו על נתוני האימון.

In [11]:
model = LogisticRegressionScratch(learning_rate=0.5, n_iterations=300, lambda_reg=0.01)
model.fit(X_train, y_train)
print("Training done.")

Training done.


## חלק 5 - חיזוי ושערוך איכות המודל על ה-test set 
<br>
בשלב זה מפעילים את המודל על נתוני הבדיקה ומשווים בין מספר תחזיות לבין התוצאות האמיתיות  
<br>
**F1** כדי להעריך את איכות הסיווג ואת ביצועי המודל על נתונים שלא שימשו בתהליך האימון, בסיום מחשבים את מדד ה־

In [12]:
test_predictions = model.predict(X_test)

print("First 5 predictions on test set:")
for i in range(5):
    predicted = "positive" if test_predictions[i] == 1 else "negative"
    actual = "positive" if y_test[i] == 1 else "negative"
    print(f"Review {i}: predicted={predicted}, actual={actual}")

macro_f1 = f1_score(y_test, test_predictions, average="macro")
print(f"\nMacro-average F1 on test set: {macro_f1:.4f}")

First 5 predictions on test set:
Review 0: predicted=negative, actual=negative
Review 1: predicted=positive, actual=negative
Review 2: predicted=positive, actual=positive
Review 3: predicted=negative, actual=negative
Review 4: predicted=negative, actual=negative

Macro-average F1 on test set: 0.8154
